# Official CODI endpoint rank-matched retention experiment

This run compares energy, answer-conditioned, and parameter-aware endpoint selectors at exactly rank 3 in states 11 and 12. It trains selected-only and discarded-complement arms, evaluates all arms on full GSM8K with three training seeds, performs paired hierarchical bootstrap analysis, and exports resumable checksummed results. Enable Internet and a T4-or-newer GPU, attach the three completed experiment datasets, then use **Save Version → Save & Run All**.

## 1. Frozen run configuration

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the immutable commit after pushing.
REPO_DIR = "/kaggle/working/latent-reasoning"

REPRODUCTION_SUMMARY_INPUT = ""
RESUME_INPUT = ""  # Optional previous endpoint-retention export root.
ENERGY_BASIS_INPUT = ""  # Optional explicit /kaggle/input/.../basis.pt override.
ANSWER_CONDITIONED_BASIS_INPUT = ""
PARAMETER_AWARE_BASIS_INPUT = ""
RUN_REPRODUCTION_GATE_IF_MISSING = True
RUN_SMOKE = True
RUN_FULL = True

TRAINING_EXAMPLES = 512
DATA_SEED = 53
TRAINING_SEEDS = [53, 59, 61]
EPOCHS = 1
BATCH_SIZE = 4
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.0
EVAL_BATCH_SIZE = 128
PRECISION = "float32"
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 0
NONINFERIORITY_MARGIN = 0.01
ARMS = ["answer_only", "full_common", "energy_selected", "answer_conditioned_selected", "parameter_aware_selected", "energy_complement", "answer_conditioned_complement", "parameter_aware_complement"]

UPLOAD_AS_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = "jonraza15/official-codi-endpoint-retention"

## 2. Install and pin the repository

In [ ]:
import datetime, hashlib, json, os, pathlib, shutil, subprocess, sys
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")
repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main": print("PIN RUN_COMMIT BEFORE THE FINAL RUN:", commit)

## 3. Hardware and implementation tests

In [ ]:
import torch, transformers
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
capability = torch.cuda.get_device_capability(0)
print("Torch:", torch.__version__, "Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0), "capability:", capability)
assert capability >= (7, 0), "Use a T4 or newer GPU"
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_endpoint_retention.py", "tests/test_endpoint_tsvc_corrected.py", "tests/test_endpoint_answer_conditioned.py", "tests/test_endpoint_parameter_aware.py", "tests/test_official_codi_target_utility.py"], cwd=REPO_DIR, check=True)

## 4. Locate and verify all three frozen selector artifacts

In [ ]:
EXPECTED_CONTRACTS = {"source_faithful_student_and_teacher_answer_colon_v2": "energy", "answer_conditioned_colon_block_states_v1": "answer_conditioned", "parameter_aware_colon_final_two_blocks_v1": "parameter_aware"}
EXPLICIT_BASIS_INPUTS = {"energy": ENERGY_BASIS_INPUT, "answer_conditioned": ANSWER_CONDITIONED_BASIS_INPUT, "parameter_aware": PARAMETER_AWARE_BASIS_INPUT}
def merged_metadata(path, payload):
    metadata = dict(payload.get("metadata", {})); manifest_path = path.parent / "run_manifest.json"; parity_path = path.parent / "native_loss_gradient_parity.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
        for key, value in manifest.items(): metadata.setdefault(key, value)
    if parity_path.is_file() and "native_parity_gate" not in metadata: metadata["native_parity_gate"] = json.loads(parity_path.read_text())
    return metadata
basis_candidates = list(pathlib.Path("/kaggle/input").rglob("basis.pt"))
basis_candidates_by_method = {method: [] for method in EXPECTED_CONTRACTS.values()}; diagnostics = []
for path in basis_candidates:
    try: payload = torch.load(path, map_location="cpu", weights_only=False); metadata = merged_metadata(path, payload)
    except Exception as error: diagnostics.append((str(path), "unreadable", type(error).__name__)); continue
    contract = metadata.get("contract"); diagnostics.append((str(path), contract, metadata.get("calibration_examples"), metadata.get("residual_fit_examples"), metadata.get("native_parity_gate", {}).get("status")))
    if contract in EXPECTED_CONTRACTS:
        method = EXPECTED_CONTRACTS[contract]
        is_full = (method == "energy" and metadata.get("calibration_examples") == 5000) or (method != "energy" and metadata.get("residual_fit_examples") == 1024 and metadata.get("direction_selection_examples") == 1024)
        if is_full and metadata.get("native_parity_gate", {}).get("status") == "passed": basis_candidates_by_method[method].append(path)
basis_by_method = {}
for method, explicit in EXPLICIT_BASIS_INPUTS.items():
    paths = [pathlib.Path(explicit)] if explicit else basis_candidates_by_method[method]
    if len(paths) != 1:
        print("Mounted Kaggle input roots:", [str(path) for path in pathlib.Path("/kaggle/input").iterdir()]); print("basis.pt diagnostics:", *diagnostics, sep="\n  " )
        raise AssertionError(f"Need one full completed {method} basis, found {paths}. Attach the corresponding completed Kaggle dataset or set its explicit *_BASIS_INPUT path.")
    basis_by_method[method] = paths[0]
ENERGY_BASIS = basis_by_method["energy"]
ANSWER_CONDITIONED_BASIS = basis_by_method["answer_conditioned"]
PARAMETER_AWARE_BASIS = basis_by_method["parameter_aware"]
for method, path in basis_by_method.items(): print(method, path, hashlib.sha256(path.read_bytes()).hexdigest())

## 5. Durable paths, logs, and resume

In [ ]:
OUTPUT_ROOT = repo / "outputs" / "official_codi_endpoint_retention"
REPORT_ROOT = repo / "reports" / "official_codi_endpoint_retention"
LOG_ROOT = repo / "logs" / "official_codi_endpoint_retention"
VALIDATION_ROOT = repo / "outputs" / "official_codi_gpt2"
for path in (OUTPUT_ROOT, REPORT_ROOT, LOG_ROOT, VALIDATION_ROOT): path.mkdir(parents=True, exist_ok=True)
def run_persisted(command, log_name):
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, command)), flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, command))} ===\n")
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True); log.write(line)
        code = process.wait()
    if code != 0: raise RuntimeError(f"Command failed with exit code {code}; inspect {log_path}")
    return log_path
if RESUME_INPUT:
    candidates = list(pathlib.Path(RESUME_INPUT).rglob("official_codi_endpoint_retention"))
    candidates = [p for p in candidates if p.is_dir() and (p / "runs").exists()]
    assert candidates, "No endpoint-retention output tree found"
    shutil.copytree(sorted(candidates, key=lambda p: (len(p.parts), p.as_posix()))[0], OUTPUT_ROOT, dirs_exist_ok=True)
    print("Restored resumable outputs")

## 6. Locate or run the official full-GSM8K reproduction gate

In [ ]:
EXPECTED_REVISION = "fd641b3d3edc59e4f534b55588e906588c9e36bb"
def passed_summary(path):
    try: payload = json.loads(path.read_text())
    except Exception: return False
    gate = payload.get("accuracy_gate", payload.get("gate")); status = gate.get("status") if isinstance(gate, dict) else gate
    return status == "passed" and payload.get("evaluated_counts", {}).get("gsm8k") == 1319 and payload.get("checkpoint_revision") in {None, EXPECTED_REVISION}
if REPRODUCTION_SUMMARY_INPUT:
    REPRODUCTION_SUMMARY = pathlib.Path(REPRODUCTION_SUMMARY_INPUT); assert passed_summary(REPRODUCTION_SUMMARY)
else:
    candidates = [p for p in pathlib.Path("/kaggle/input").rglob("summary.json") if passed_summary(p)] + [p for p in VALIDATION_ROOT.rglob("summary.json") if passed_summary(p)]
    if not candidates:
        assert RUN_REPRODUCTION_GATE_IF_MISSING
        run_persisted([sys.executable, "-u", "-m", "src.eval.official_codi", "--config", "configs/official_codi_gpt2.yaml", "--datasets", "gsm8k", "--limit", "0", "--device", "cuda", "--output-dir", str(VALIDATION_ROOT)], "official_codi_gsm8k_gate.log")
        candidates = [p for p in VALIDATION_ROOT.rglob("summary.json") if passed_summary(p)]
    assert candidates; REPRODUCTION_SUMMARY = sorted(candidates, key=lambda p: p.as_posix())[0]
print("Reproduction summary:", REPRODUCTION_SUMMARY)

## 7. One-arm command and mandatory smoke

In [ ]:
def retention_command(root, arm, training_seed, training_examples=TRAINING_EXAMPLES, eval_limit=0):
    return [sys.executable, "-u", "scripts/run_official_codi_endpoint_retention.py", "--config", "configs/official_codi_gpt2.yaml", "--reproduction-summary", str(REPRODUCTION_SUMMARY), "--energy-basis", str(ENERGY_BASIS), "--answer-conditioned-basis", str(ANSWER_CONDITIONED_BASIS), "--parameter-aware-basis", str(PARAMETER_AWARE_BASIS), "--output-dir", str(root), "--arm", arm, "--training-seed", str(training_seed), "--data-seed", str(DATA_SEED), "--training-examples", str(training_examples), "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--learning-rate", str(LEARNING_RATE), "--weight-decay", str(WEIGHT_DECAY), "--save-every", "32", "--eval-limit", str(eval_limit), "--eval-batch-size", str(EVAL_BATCH_SIZE), "--precision", PRECISION, "--device", "cuda"]
if RUN_SMOKE:
    for arm in ("energy_selected", "energy_complement"):
        root = OUTPUT_ROOT / "smoke" / arm
        run_persisted(retention_command(root, arm, 53, training_examples=8, eval_limit=16), f"smoke_{arm}.log")
        summary = json.loads((root / "summary.json").read_text()); assert summary["evaluation"]["count"] == 16
    print("Selected/complement smoke path complete")

## 8. Run all 24 full-GSM8K arms (resumable)

In [ ]:
RUNS_ROOT = OUTPUT_ROOT / "runs"
if RUN_FULL:
    for training_seed in TRAINING_SEEDS:
        for arm in ARMS:
            root = RUNS_ROOT / f"seed_{training_seed}" / arm
            run_persisted(retention_command(root, arm, training_seed), f"seed{training_seed}_{arm}.log")
summaries = list(RUNS_ROOT.rglob("summary.json"))
print("Completed full runs:", len(summaries), "/", len(TRAINING_SEEDS) * len(ARMS))
if RUN_FULL: assert len(summaries) == len(TRAINING_SEEDS) * len(ARMS)

## 9. Paired hierarchical accuracy analysis

In [ ]:
REPORT_PATH = REPORT_ROOT / "endpoint_retention_summary.json"
if RUN_FULL:
    run_persisted([sys.executable, "-u", "scripts/analyze_official_codi_endpoint_retention.py", "--runs-root", str(RUNS_ROOT), "--output", str(REPORT_PATH), "--bootstrap-samples", str(BOOTSTRAP_SAMPLES), "--bootstrap-seed", str(BOOTSTRAP_SEED), "--noninferiority-margin", str(NONINFERIORITY_MARGIN)], "analyze_endpoint_retention.log")
    report = json.loads(REPORT_PATH.read_text())
    print("Highest selected accuracy:", report["highest_selected_accuracy"])
    for method, result in report["selected_vs_full_common"].items(): print(method, "vs full (pp)=", result["delta_percentage_points"], "CI=", result["bootstrap_95_ci_percentage_points"], "noninferior=", result["noninferior_to_full_common"])
    for method, result in report["selected_vs_own_complement"].items(): print(method, "selected minus complement (pp)=", result["delta_percentage_points"], "CI=", result["bootstrap_95_ci_percentage_points"])
    for method, result in report["selected_vs_answer_only"].items(): print(method, "selected minus answer-only (pp)=", result["delta_percentage_points"], "CI=", result["bootstrap_95_ci_percentage_points"])
    print(report["inference_speed_interpretation"]["reason"])

## 10. Build checksummed resumable export

In [ ]:
EXPORT_ROOT = pathlib.Path("/kaggle/working/official_codi_endpoint_retention_export")
if EXPORT_ROOT.exists(): shutil.rmtree(EXPORT_ROOT)
export_repo = EXPORT_ROOT / "latent-reasoning"
shutil.copytree(OUTPUT_ROOT, export_repo / "outputs" / "official_codi_endpoint_retention")
if REPORT_ROOT.exists(): shutil.copytree(REPORT_ROOT, export_repo / "reports" / "official_codi_endpoint_retention")
shutil.copytree(LOG_ROOT, export_repo / "logs" / "official_codi_endpoint_retention")
validation_export = export_repo / "outputs" / "official_codi_gpt2_reproduction"; validation_export.mkdir(parents=True, exist_ok=True)
shutil.copy2(REPRODUCTION_SUMMARY, validation_export / "summary.json")
(EXPORT_ROOT / "RUN_COMMIT.txt").write_text(commit + "\n")
(EXPORT_ROOT / "RUN_INSTRUCTIONS.txt").write_text("Attach this dataset and set RESUME_INPUT to its root. Reattach all three immutable source basis datasets.\n")
files = sorted(path for path in EXPORT_ROOT.rglob("*") if path.is_file())
(EXPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.relative_to(EXPORT_ROOT).as_posix()}" for path in files) + "\n")
print("Export root:", EXPORT_ROOT, "files:", len(list(EXPORT_ROOT.rglob("*"))))
if UPLOAD_AS_KAGGLE_DATASET:
    import kagglehub
    kagglehub.dataset_upload(KAGGLE_DATASET_HANDLE, str(EXPORT_ROOT), version_notes=f"Rank-matched endpoint retention at {commit}")

## Interpretation rule

A selector succeeds only if selected-only is non-inferior to the full target and is better than its own complement. Similar selected-only and answer-only accuracy means the auxiliary target was unnecessary, not that compression succeeded. Timed generation is a guardrail only: the projection is used during training, so this unchanged 12-block, width-768 model cannot gain inference speed from top-k selection.